[Reference](https://rajuhemanth456.medium.com/llamaindex-in-python-a-hands-on-rag-guide-b95e434e254d)

# Installing LlamaIndex
```
# Create a virtual environment (recommended)
python -m venv venv
source venv/bin/activate  # on Windows use: venv\Scripts\activate
# Install LlamaIndex core
pip install llama-index
# Optionally, install an LLM integration (example: OpenAI)
pip install llama-index-llms-openai
# Optional: a Hugging Face embedding model
pip install llama-index-embeddings-huggingface
```

```
export OPENAI_API_KEY="your_api_key_here"
```

# Indices & Vector Stores
Chroma, Qdrant, Pinecone, Milvus, Cassandra/Astra DB

# LLMs, embeddings & Settings

In [1]:
from llama_index.core import Settings
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
Settings.llm = OpenAI(model="gpt-4o")  # or any supported model
Settings.embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-small-en-v1.5"
)

# Retrievers, Query Engines & Chat Engines
- A Retriever picks relevant nodes given a query (VectorIndexRetriever, etc.).
- A Query Engine orchestrates retrieval + LLM response synthesis.
- A Chat Engine adds conversation state on top (multiple back-and-forth turns).

In [2]:
from llama_index.core import VectorStoreIndex, Document
# 1. Create some in-memory documents
docs = [
    Document(
        text=(
            "LlamaIndex is a Python framework that helps you build "
            "retrieval-augmented generation (RAG) apps by connecting "
            "your data (documents, APIs, DBs) to LLMs."
        )
    ),
    Document(
        text=(
            "RAG works by retrieving relevant context from a knowledge "
            "base and passing it into the LLM along with the user question."
        )
    ),
]
# 2. Build a vector index over the docs
index = VectorStoreIndex.from_documents(docs)
# 3. Turn the index into a query engine
query_engine = index.as_query_engine()
# 4. Ask a question!
response = query_engine.query(
    "Explain what LlamaIndex does, in one sentence."
)
print(response)

# RAG over local files with SimpleDirectoryReader

In [3]:
import os
from llama_index.core import (
    VectorStoreIndex,
    SimpleDirectoryReader,
    Settings,
)
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
# --- 1. Configure models (global Settings) ---
os.environ["OPENAI_API_KEY"] = "your_key_here"  # or use env vars
Settings.llm = OpenAI(model="gpt-4o")  # or "gpt-3.5-turbo"
Settings.embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-small-en-v1.5"
)
# --- 2. Load documents from a directory ---
# Put some .txt / .md / .pdf files in ./data/docs
documents = SimpleDirectoryReader("./data/docs").load_data()
print(f"Loaded {len(documents)} documents")
# --- 3. Create the vector index ---
index = VectorStoreIndex.from_documents(documents)
# --- 4. Create a query engine ---
query_engine = index.as_query_engine(similarity_top_k=5)
# --- 5. Ask questions about your docs ---
while True:
    q = input("\nAsk something about your docs (or 'q'): ")
    if q.lower().strip() == "q":
        break
    response = query_engine.query(q)
    print("\nAnswer:\n", response, "\n")

# Customizing retrieval (top_k, similarity cutoff, filters)

In [4]:
from llama_index.core import (
    VectorStoreIndex,
    SimpleDirectoryReader,
    get_response_synthesizer,
)
from llama_index.core.retrievers import VectorIndexRetriever
from llama_index.core.query_engine import RetrieverQueryEngine
# 1. Load & index data (same as before)
documents = SimpleDirectoryReader("./data/docs").load_data()
index = VectorStoreIndex.from_documents(documents)
# 2. Build a custom retriever
retriever = VectorIndexRetriever(
    index=index,
    similarity_top_k=8,        # pull more candidates
)
# 3. Optionally, customize response synthesis
response_synthesizer = get_response_synthesizer()
# 4. Build a query engine using our retriever
query_engine = RetrieverQueryEngine(
    retriever=retriever,
    response_synthesizer=response_synthesizer,
)
# 5. Query
response = query_engine.query(
    "Give me a concise summary of how our billing system works."
)
print(response)

# Turning RAG into a chat experience

In [5]:
from llama_index.core import (
    VectorStoreIndex,
    SimpleDirectoryReader,
)
# Assuming Settings.llm and Settings.embed_model are set
documents = SimpleDirectoryReader("./data/docs").load_data()
index = VectorStoreIndex.from_documents(documents)
# Context-based chat engine: retrieve context per question
chat_engine = index.as_chat_engine(
    chat_mode="context",  # common default for RAG-style chat
    similarity_top_k=5,
)
print("Ask questions about your docs. Type 'exit' to quit.")
while True:
    user_input = input("\nYou: ")
    if user_input.lower() in ("exit", "quit"):
        break
    response = chat_engine.chat(user_input)
    print("\nBot:", response)

#Using Ollama + Hugging Face embeddings

In [6]:
from llama_index.core import (
    Settings,
    VectorStoreIndex,
    SimpleDirectoryReader,
)
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
# Configure local Llama 3 via Ollama
Settings.llm = Ollama(model="llama3")
# Use a BGE embedding model
Settings.embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-small-en-v1.5"
)
# Load docs and create index
documents = SimpleDirectoryReader("./data/docs").load_data()
index = VectorStoreIndex.from_documents(documents)
query_engine = index.as_query_engine()
print(query_engine.query("What is this folder about?"))